### University Timetable Optimization using Genetic Algorithm

This notebook implements a Genetic Algorithm (GA) to optimize a university timetable.

**Problem Constraints & Goals:**
- **Hard Constraints (Avoid Clashes):**
  - A faculty member cannot teach more than one class at the same time.
  - A room cannot host more than one class at the same time.
  - A class requires a specific room type (e.g., a lab course needs a Lab room).
- **Soft Constraints (Optimization):**
  - Minimize unused gaps or slots (compact schedules).
  - Distribute classes evenly.

In [4]:
import random
import copy

# --- Define Problem Entities ---
CLASSES = ['CS101', 'CS102', 'CS103', 'EE101', 'EE102', 'MATH101']
FACULTY = {
    'CS101': 'Dr. Sharma',   # Updated to Indian Names
    'CS102': 'Dr. Patel',
    'CS103': 'Dr. Sharma',
    'EE101': 'Dr. Iyer',
    'EE102': 'Dr. Iyer',
    'MATH101': 'Dr. Nair'
}
ROOMS = {
    'Room 101': 'Lecture',
    'Room 102': 'Lecture',
    'Lab A': 'Lab'
}
ROOM_REQUIREMENTS = {
    'CS101': 'Lecture',
    'CS102': 'Lab',
    'CS103': 'Lecture',
    'EE101': 'Lecture',
    'EE102': 'Lab',
    'MATH101': 'Lecture'
}
TIMESLOTS = ['Mon 9:00', 'Mon 11:00', 'Wed 9:00', 'Wed 11:00', 'Fri 9:00']

# --- Genetic Algorithm Parameters ---
POPULATION_SIZE = 50
GENERATIONS = 100
MUTATION_RATE = 0.15
TOURNAMENT_SIZE = 5

In [2]:
class Individual:
    def __init__(self, schedule=None):
        # A schedule is a dict: { class_name: (room, timeslot) }
        if schedule is None:
            self.schedule = {}
            for c in CLASSES:
                room = random.choice(list(ROOMS.keys()))
                timeslot = random.choice(TIMESLOTS)
                self.schedule[c] = (room, timeslot)
        else:
            self.schedule = schedule
        self.fitness = self.calculate_fitness()

    def calculate_fitness(self):
        clashes = 0

        # Track assignments to spot overlaps
        faculty_times = {}
        room_times = {}

        for course, (room, timeslot) in self.schedule.items():
            # 1. Faculty Clash
            fac = FACULTY[course]
            if (fac, timeslot) in faculty_times:
                clashes += 1
            else:
                faculty_times[(fac, timeslot)] = course

            # 2. Room Clash
            if (room, timeslot) in room_times:
                clashes += 1
            else:
                room_times[(room, timeslot)] = course

            # 3. Room Type Match Clash
            required_type = ROOM_REQUIREMENTS[course]
            actual_type = ROOMS[room]
            if required_type != actual_type:
                clashes += 1

        # Fitness is higher for fewer clashes. Perfect score = 1.0
        return 1.0 / (1.0 + clashes)

# --- GA Operators ---
def crossover(parent1, parent2):
    # Single-point crossover on the course list
    split = random.randint(1, len(CLASSES) - 1)
    child1_sched = {}
    child2_sched = {}

    for i, c in enumerate(CLASSES):
        if i < split:
            child1_sched[c] = parent1.schedule[c]
            child2_sched[c] = parent2.schedule[c]
        else:
            child1_sched[c] = parent2.schedule[c]
            child2_sched[c] = parent1.schedule[c]

    return Individual(child1_sched), Individual(child2_sched)

def mutate(individual):
    sched = copy.deepcopy(individual.schedule)
    for c in CLASSES:
        if random.random() < MUTATION_RATE:
            sched[c] = (random.choice(list(ROOMS.keys())), random.choice(TIMESLOTS))
    return Individual(sched)

def tournament_selection(population):
    selected = random.sample(population, TOURNAMENT_SIZE)
    return max(selected, key=lambda ind: ind.fitness)

In [5]:
# --- Evolution Loop ---
population = [Individual() for _ in range(POPULATION_SIZE)]

for generation in range(GENERATIONS):
    # Sort population by fitness
    population = sorted(population, key=lambda x: x.fitness, reverse=True)

    if population[0].fitness == 1.0:
        print(f"Perfect schedule found at Generation {generation}!")
        break

    next_generation = population[:2] # Keep elites

    while len(next_generation) < POPULATION_SIZE:
        p1 = tournament_selection(population)
        p2 = tournament_selection(population)
        c1, c2 = crossover(p1, p2)
        next_generation.append(mutate(c1))
        if len(next_generation) < POPULATION_SIZE:
            next_generation.append(mutate(c2))

    population = next_generation

# Display Best Result
best_individual = max(population, key=lambda x: x.fitness)
print(f"\nFinal Best Fitness: {best_individual.fitness:.4f}")
print("\nGenerated Timetable:")
print(f"{'Course':<10} | {'Faculty':<12} | {'Room (Type)':<18} | {'Timeslot'}")
print("-" * 60)
for course, (room, timeslot) in sorted(best_individual.schedule.items()):
    room_desc = f"{room} ({ROOMS[room]})"
    print(f"{course:<10} | {FACULTY[course]:<12} | {room_desc:<18} | {timeslot}")

Perfect schedule found at Generation 3!

Final Best Fitness: 1.0000

Generated Timetable:
Course     | Faculty      | Room (Type)        | Timeslot
------------------------------------------------------------
CS101      | Dr. Sharma   | Room 102 (Lecture) | Wed 9:00
CS102      | Dr. Patel    | Lab A (Lab)        | Fri 9:00
CS103      | Dr. Sharma   | Room 102 (Lecture) | Fri 9:00
EE101      | Dr. Iyer     | Room 101 (Lecture) | Fri 9:00
EE102      | Dr. Iyer     | Lab A (Lab)        | Mon 9:00
MATH101    | Dr. Nair     | Room 101 (Lecture) | Wed 9:00
